## 05 — Evaluation
Runs the evaluation questions from `config/eval_questions.yml` with and without retrieval,
measures retrieval hit rate, "I don't know" behaviour, keyword correctness and groundedness,
and writes the results to `rag_eval_results`.

In [0]:
%pip install databricks-ai-search openai

In [0]:
dbutils.library.restartPython()

In [0]:
for key in ["catalog", "silver_schema", "gold_schema", "rag_schema"]:
    dbutils.widgets.text(key, "")

In [0]:
%run ./rag_functions

In [0]:
import os
import yaml

eval_path = os.path.abspath(os.path.join(os.getcwd(), "..", "config", "eval_questions.yml"))

with open(eval_path, "r") as f:
    eval_questions = yaml.safe_load(f)["questions"]

logger.info(
    "Loaded %s evaluation questions: %s",
    len(eval_questions),
    {t: sum(1 for q in eval_questions if q["type"] == t) for t in ("specific", "general", "unanswerable")},
)

In [0]:
import uuid
from datetime import datetime, timezone

run_id = str(uuid.uuid4())
run_at = datetime.now(timezone.utc)
raw_results = []

for item in eval_questions:
    for use_retrieval in (True, False):
        try:
            result = answer_question(item["question"], use_retrieval=use_retrieval, caller="evaluation")
            error = None
        except Exception as exc:
            result = None
            error = str(exc)[:500]
            logger.warning("Question %s (retrieval=%s) failed: %s", item["id"], use_retrieval, error)

        raw_results.append({"item": item, "use_retrieval": use_retrieval, "result": result, "error": error})

logger.info(
    "Evaluation run %s: %s answers, %s errors",
    run_id, len(raw_results), sum(1 for r in raw_results if r["error"]),
)

In [0]:
from pyspark.sql import functions as F

IDK_PHRASE = "I don't know based on the available videos"

JUDGE_PROMPT = """You check whether an answer is supported by the given context.
On the first line reply with exactly one word:
SUPPORTED - if every factual claim in the answer appears in the context, or the answer says it does not know;
UNSUPPORTED - if the answer contains any claim that is not in the context.
On the second line give a short reason."""


def video_id_of(chunk_id: str) -> str:
    return chunk_id.rsplit("_", 1)[0]


def get_chunk_texts(chunk_ids: list[str]) -> list[str]:
    rows = (
        spark.table(chunks_table)
        .filter(F.col("chunk_id").isin(chunk_ids))
        .select("chunk_id", "chunk_text")
        .collect()
    )
    texts = {row["chunk_id"]: row["chunk_text"] for row in rows}
    return [texts[c] for c in chunk_ids if c in texts]


def judge_groundedness(answer: str, context_texts: list[str]) -> tuple[bool | None, str]:
    context = "\n\n".join(f"[{n}] {text}" for n, text in enumerate(context_texts, start=1))
    verdict = ask_llm([
        {"role": "system", "content": JUDGE_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nAnswer:\n{answer}"},
    ])["answer"].strip()

    first_line = verdict.splitlines()[0].strip().upper() if verdict else ""
    if first_line.startswith("UNSUPPORTED"):
        return False, verdict
    if first_line.startswith("SUPPORTED"):
        return True, verdict
    return None, verdict

In [0]:
rag_context = {
    r["item"]["id"]: get_chunk_texts([s["chunk_id"] for s in r["result"]["sources"]])
    for r in raw_results
    if r["use_retrieval"] and r["result"]
}

eval_rows = []
for r in raw_results:
    item, result = r["item"], r["result"]
    answer = result["answer"] if result else None
    has_expectation = bool(item["expected_video_ids"])

    retrieved_videos = [video_id_of(s["chunk_id"]) for s in result["sources"]] if result else []
    retrieval_hit = (
        any(v in retrieved_videos for v in item["expected_video_ids"])
        if r["use_retrieval"] and has_expectation and result else None
    )
    keyword_correct = (
        any(k.lower() in answer.lower() for k in item["expected_keywords"])
        if has_expectation and answer else None
    )
    said_idk = IDK_PHRASE.lower() in answer.lower() if answer else None

    grounded, judge_text = (None, None)
    if answer and rag_context.get(item["id"]):
        grounded, judge_text = judge_groundedness(answer, rag_context[item["id"]])

    eval_rows.append({
        "run_id": run_id,
        "run_at": run_at,
        "question_id": item["id"],
        "question_type": item["type"],
        "question": item["question"],
        "use_retrieval": r["use_retrieval"],
        "answer": answer,
        "retrieved_video_ids": retrieved_videos,
        "retrieval_hit": retrieval_hit,
        "keyword_correct": keyword_correct,
        "said_idk": said_idk,
        "grounded": grounded,
        "judge_reason": judge_text,
        "prompt_tokens": result["prompt_tokens"] if result else None,
        "request_id": result["request_id"] if result else None,
        "error": r["error"],
    })

logger.info("Computed metrics for %s answers", len(eval_rows))

In [0]:
from pyspark.sql.types import (
    ArrayType, BooleanType, IntegerType, StringType,
    StructField, StructType, TimestampType,
)

EVAL_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("run_at", TimestampType(), False),
    StructField("question_id", StringType(), False),
    StructField("question_type", StringType(), False),
    StructField("question", StringType(), False),
    StructField("use_retrieval", BooleanType(), False),
    StructField("answer", StringType(), True),
    StructField("retrieved_video_ids", ArrayType(StringType()), True),
    StructField("retrieval_hit", BooleanType(), True),
    StructField("keyword_correct", BooleanType(), True),
    StructField("said_idk", BooleanType(), True),
    StructField("grounded", BooleanType(), True),
    StructField("judge_reason", StringType(), True),
    StructField("prompt_tokens", IntegerType(), True),
    StructField("request_id", StringType(), True),
    StructField("error", StringType(), True),
])

eval_df = spark.createDataFrame(eval_rows, schema=EVAL_SCHEMA)
eval_df.write.mode("append").saveAsTable(eval_results_table)

logger.info("Wrote %s evaluation rows for run %s to %s", eval_df.count(), run_id, eval_results_table)

In [0]:
def rate(column: str):
    return F.round(F.avg(F.col(column).cast("int")), 2).alias(f"{column}_rate")


summary_df = (
    spark.table(eval_results_table)
    .filter(F.col("run_id") == run_id)
    .groupBy("question_type", "use_retrieval")
    .agg(
        F.count("*").alias("answers"),
        rate("retrieval_hit"),
        rate("keyword_correct"),
        rate("said_idk"),
        rate("grounded"),
        F.round(F.avg("prompt_tokens")).alias("avg_prompt_tokens"),
    )
    .orderBy("question_type", F.col("use_retrieval").desc())
)

display(summary_df)